# Session02: 大規模言語モデルの医学画像分析質問への対応精度評価

## Motivation（モチベーション）

この演習は、以下を学ぶことを目的としています：

- 医学画像に関する多肢選択問題を使って、モデルの実務的な性能を評価する方法を身につける。
- CSV データの前処理、集計、表・図の作成を通じて再現可能な解析ワークフローを学ぶ。
- 人間の受験者（student baseline）と LLM の結果を比較し、モデルの強み・弱みや分野依存性を理解する。
- コードを書き、可視化や統計指標の解釈を行うことで論文の Results および Materials and Methods セクションを構築する練習をする。


## 一般的な注意（単位・有効数字）

- 単位は原則として SI 単位で統一する（例: `mm`, `cm`, `mL`, `%`）。
- 数値と単位の間には半角スペースを入れる（例: `10 mm`）。ただし `%` は `76.0%` のように続けて表記する。
- 比較する値は同じ桁数で示し、有効数字をそろえる。
- 平均と標準偏差は桁数をそろえて記載する（例: `72.9% (SD = 9.3%)`）。
- 過度に細かい桁は避け、測定精度・解析目的に見合う桁数に丸める。

以下のノートブックは自分でコードを書きながら解析を進められるよう、実装のステップを記載しています。Google Colab用のリンクは後でスクリプトで差し替えてください。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar3_Public/blob/main/notebooks/session02_materials_and_results.ipynb)

## Colaboを開いた後の手順

data, imagesフォルダをそのままColaboのフォルダにコピーしてください。中身は手本の画像およびダミーデータです。自分で作成した表・図は、提出用として `./save_figures` フォルダに保存してください。保存するファイル名は `Table_1.png`, `Table_2.png`, `Figure_1.png`, `Figure_2.png` に統一します。

## 1) data の中身について (説明)

このノートブックでは、実際の 2024 年放射線診断学専門医試験の問題をもとに、LLM を API で呼び出して解かせた結果を、少し変更したダミーデータを扱います。データファイル: `markdowns/data/dummy_LLM.csv` を読み込んでください。

主なカラムは次の通りです:
- `Subspeciality`: 問題が属する医学専門分野 (例: Neuroradiology & Head & Neck)
- `Answer`: 正解ラベル (多肢選択の正解)
- `*_ans` カラム群: 各モデルが出力した候補回答 (例: `GPT-4.1_ans`)
- このCSVには正解/不正解の `0/1` 列は含めていません。`Answer` と各 `*_ans` を比較して、モデルごとの正解判定列（1=正解, 0=不正解）を自分で作成してください。

注意: 実データを用いるときは、`pd.read_csv` で読み込み、列名とデータ型を必ず確認してください。


In [18]:
# TODO: データ読み込みと前処理を実装してください
# 1) pandas を import する
import pandas as pd
import os

# 2) CSV を読み込む（例: dummy_LLM.csv）
df = pd.read_csv('/content/data/dummy_LLM.csv')

# 3) カラム名とデータ型を確認する
print("Original DataFrame Info:")
print(df.info())

# 4) 欠損値・余分な空白を処理する
# 欠損値の処理 (ここでは欠損値を含む行を削除する例)
initial_rows = len(df)
df.dropna(inplace=True)
print(f"\nDropped {initial_rows - len(df)} rows with missing values.")

# 余分な空白の処理 (文字列型のカラムに対して適用)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()
print("Trimmed leading/trailing whitespaces from object columns.")

print("\nDataFrame Info after preprocessing:")
print(df.info())

# 5) Answer と *_ans を比較して、各モデルの正解判定列（1/0）を作成する
# モデルの回答カラムを特定
model_ans_cols = [col for col in df.columns if col.endswith('_ans')]

# 各モデルの正解判定列を作成
for col in model_ans_cols:
    model_name = col.replace('_ans', '') # モデル名を抽出
    df[model_name] = (df[col] == df['Answer']).astype(int)
    print(f"Created column '{model_name}' for correctness (1/0).")

# 新しく作成された判定列を含むDataFrameの先頭5行を表示
print("\nDataFrame with correctness columns:")
display(df.head())

Original DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Subspeciality                          100 non-null    object
 1   Answer                                 100 non-null    object
 2   GPT-4.1_ans                            100 non-null    object
 3   o3_ans                                 100 non-null    object
 4   Claude 3.7 Sonnet_ans                  100 non-null    object
 5   Claude 3.7 Sonnet-thinking_ans         100 non-null    object
 6   Gemini 2.5 Pro Preview_ans             100 non-null    object
 7   Gemini 2.5 Flash Preview-thinking_ans  100 non-null    object
dtypes: object(8)
memory usage: 6.4+ KB
None

Dropped 0 rows with missing values.
Trimmed leading/trailing whitespaces from object columns.

DataFrame Info after preprocessing:
<class 'pandas.core.frame

,Subspeciality,Answer,GPT-4.1_ans,o3_ans,Claude 3.7 Sonnet_ans,Claude 3.7 Sonnet-thinking_ans,Gemini 2.5 Pro Preview_ans,Gemini 2.5 Flash Preview-thinking_ans,GPT-4.1,o3,Claude 3.7 Sonnet,Claude 3.7 Sonnet-thinking,Gemini 2.5 Pro Preview,Gemini 2.5 Flash Preview-thinking
0,Neuroradiology & Head & Neck,d,d,a,a,a,a,a,1,0,0,0,0,0
1,Neuroradiology & Head & Neck,b,b,b,b,b,b,b,1,1,1,1,1,1
2,Neuroradiology & Head & Neck,a,c,c,c,c,c,c,0,0,0,0,0,0
3,Neuroradiology & Head & Neck,d,d,d,e,e,d,a,1,1,0,0,1,0
4,Neuroradiology & Head & Neck,e,e,e,e,e,c,e,1,1,1,1,0,1


## 2) 各分野の問題数を数えて Table をつくる

目標: 各 `Subspeciality` ごとの問題数を集計し、表（Table 1）を作成します。

次の手順を参考に進めてください。
- `df['Subspeciality'].value_counts()` で各分野の件数を取得します。
- 結果を `pd.DataFrame` に変換し、列名を `['Subspeciality', 'count']` のように整えます。
- 表を markdown に埋め込むには `df.to_markdown()` を使うか、表を画像として保存します。
- 作成した Table 1 は `./save_figures/Table_1.png` として保存します。
- ノートブック内で確認するときは、Table 1 のキャプションとともに `./save_figures/Table_1.png` を表示します。

In [19]:
# TODO: Table 1（分野ごとの問題数）を作成して保存してください
# ヒント: value_counts() -> DataFrame化 -> 列名を整える
# ヒント: 表を画像として保存（保存先: ./save_figures/Table_1.png）

import matplotlib.pyplot as plt
import os

# 'save_figures' ディレクトリが存在しない場合は作成
save_dir = './save_figures'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 各分野の問題数を集計
table1_df = df['Subspeciality'].value_counts().reset_index()
table1_df.columns = ['Subspeciality', 'count']

# 全体合計を計算
total_count = table1_df['count'].sum()

# 各分野の割合を計算
table1_df['percentage'] = (table1_df['count'] / total_count * 100).round(1)

# 'Total' 行を追加
total_row = pd.DataFrame([['Total', total_count, 100.0]], columns=['Subspeciality', 'count', 'percentage'])
table1_df = pd.concat([table1_df, total_row], ignore_index=True)


print("Table 1: Distribution of questions by subspeciality")
display(table1_df)

# 表を画像として保存
# Matplotlib を使用して表を描画
fig, ax = plt.subplots(figsize=(8, 4))  # 適切なサイズに調整
ax.axis('off')  # 軸を非表示にする
ax.table(cellText=table1_df.values,
         colLabels=table1_df.columns,
         loc='center',
         cellLoc='center')

plt.title('Table 1. Distribution of questions by subspeciality', loc='center', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'Table_1.png'), bbox_inches='tight', dpi=300)
plt.close(fig)

print(f"Table 1 saved to {os.path.join(save_dir, 'Table_1.png')}")

Table 1: Distribution of questions by subspeciality


,Subspeciality,count,percentage
0,Abdominal & Pelvic,27,27.0
1,Thoracic & Cardiac,20,20.0
2,Nuclear Medicine,20,20.0
3,Neuroradiology & Head & Neck,16,16.0
4,MSK & Breast,11,11.0
5,Other,6,6.0
6,Total,100,100.0


Table 1 saved to ./save_figures/Table_1.png


### Example: 質問数分布表の挿入例

以下は見本画像 `questions_table.png` を埋め込む例です。自分で作成した Table 1 は `./save_figures/Table_1.png` として保存してください。

![質問数分布表](https://github.com/Nakaura-T/DS_Seminar3_Public/blob/main/notebooks/images/questions_table.png?raw=1)

## 3) accuracy を求める

目標: 各モデルの正解率（accuracy）を定義し、全体と分野別に計算して Table 2 を作成します。

次の手順を参考に進めてください。
- まず `Answer` と各 `*_ans` を比較して、モデルごとの正解フラグ列（1=正解, 0=不正解）を作成します。
- 例: `GPT-4.1_ans` と `Answer` が一致したら `GPT-4.1=1`、不一致なら `0`。
- 正解フラグ列が作れたら、モデルごとの accuracy は `df[model].mean()` で求められます。
- パーセンテージ表示にする場合は `*100` を使います。
- 分野別 accuracy は `df.groupby('Subspeciality')[model].mean()` で求めます。
- 表の形式は、行をモデル、列を全体と各分野としたクロス表に整えます。
- 作成した Table 2 は `./save_figures/Table_2.png` として保存します。


In [20]:
# TODO: Table 2（モデル別 accuracy）を作成して保存してください
# 1) Answer と *_ans を比較して、各モデルの正解判定列（1/0）を作る
#    -> これは前処理のステップで既に実装済みです
# 2) 全体 accuracy と Subspeciality ごとの accuracy を計算する
# 3) 表を画像として保存（保存先: ./save_figures/Table_2.png）

import matplotlib.pyplot as plt
import os

# 'save_figures' ディレクトリが存在しない場合は作成
save_dir = './save_figures'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# モデルの正解判定列を特定
model_correctness_cols = [col for col in df.columns if col not in ['Subspeciality', 'Answer'] and not col.endswith('_ans')]

# 全体 accuracy を計算
overall_accuracy = df[model_correctness_cols].mean() * 100

# 分野別 accuracy を計算
subspecialty_accuracy = df.groupby('Subspeciality')[model_correctness_cols].mean() * 100

# 結果を結合して Table 2 を作成
table2_df = pd.DataFrame(overall_accuracy).T # 転置して1行にする
table2_df.index = ['Overall Accuracy (%)']
table2_df = pd.concat([table2_df, subspecialty_accuracy])

# 表の表示
print("Table 2: Overall and Subspecialty-specific Accuracy of LLMs")
display(table2_df.round(1))

# 表を画像として保存
fig, ax = plt.subplots(figsize=(10, len(table2_df) * 0.5 + 2)) # サイズを動的に調整
ax.axis('off')
ax.table(cellText=table2_df.round(1).values,
         colLabels=table2_df.columns,
         rowLabels=table2_df.index,
         loc='center',
         cellLoc='center')

plt.title('Table 2. Overall and subspecialty-specific accuracy of large language models.', loc='center', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'Table_2.png'), bbox_inches='tight', dpi=300)
plt.close(fig)

print(f"Table 2 saved to {os.path.join(save_dir, 'Table_2.png')}")

Table 2: Overall and Subspecialty-specific Accuracy of LLMs


,GPT-4.1,o3,Claude 3.7 Sonnet,Claude 3.7 Sonnet-thinking,Gemini 2.5 Pro Preview,Gemini 2.5 Flash Preview-thinking
Overall Accuracy (%),63.0,72.0,59.0,59.0,74.0,67.0
Abdominal & Pelvic,51.9,77.8,51.9,51.9,74.1,63.0
MSK & Breast,63.6,45.5,63.6,72.7,63.6,63.6
Neuroradiology & Head & Neck,68.8,68.8,50.0,43.8,56.2,43.8
Nuclear Medicine,60.0,70.0,50.0,55.0,75.0,60.0
Other,66.7,83.3,83.3,100.0,100.0,100.0
Thoracic & Cardiac,75.0,80.0,75.0,65.0,85.0,90.0


Table 2 saved to ./save_figures/Table_2.png


### Example: 精度テーブルの挿入例

以下は見本画像 `accuracy_table.png` を埋め込む例です。自分で作成した Table 2 は `./save_figures/Table_2.png` として保存してください。

![精度テーブル](https://github.com/Nakaura-T/DS_Seminar3_Public/blob/main/notebooks/images/accuracy_table.png?raw=1)

## 4) 受験生の平均、SD を求め、LLMの正解率と比較してプロットする

目標: 受験生（Human）がいると仮定した場合の平均得点と標準偏差を求め、それを LLM の精度と同じ図に重ねて比較します。

次の手順を参考に進めてください。
- 受験生データがあればその CSV を読み込み、各受験生の正答数を問題数で割って得点率を計算します。
- 受験生データがない場合は、`df['Answer']` を基準に多数決正答を human baseline として模擬する方法も考えられます。
- 平均と SD は `students_scores.mean()` / `students_scores.std()` で求めます。
- プロットでは、LLM のバープロットに `plt.axhline(student_mean, color='k', linestyle='--')` を重ね、必要に応じて `plt.fill_between(x, student_mean-student_sd, student_mean+student_sd, alpha=0.2)` を使います。
- 作成した Figure 1 は `./save_figures/Figure_1.png` として保存します。

In [24]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# 'save_figures' ディレクトリが存在しない場合は作成
save_dir = './save_figures'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 受験生（Human）の平均と標準偏差（提示された値を使用）
student_mean = 72.9  # %
student_sd = 9.3     # % (ユーザーからの更新された値を使用)

# モデルの正解率 (overall_accuracy は既に計算済み)
# overall_accuracy は Series なので、DataFrame に変換してプロットしやすくする
llm_accuracy_df = overall_accuracy.reset_index()
llm_accuracy_df.columns = ['Model', 'Accuracy (%)']

# プロットの作成
plt.figure(figsize=(12, 7))

# LLM の正解率を棒グラフとしてプロット
sns.barplot(x='Model', y='Accuracy (%)', data=llm_accuracy_df, palette='viridis')

# 受験生平均の水平線を追加
plt.axhline(y=student_mean, color='k', linestyle='--', label=f'Student Mean ({student_mean:.1f}%)')

# 受験生標準偏差の帯を追加
plt.fill_between(x=[-0.5, len(llm_accuracy_df) - 0.5],
                 y1=[student_mean - student_sd, student_mean - student_sd],
                 y2=[student_mean + student_sd, student_mean + student_sd],
                 color='gray', alpha=0.2, label=f'Student SD (±{student_sd:.1f}%)')

plt.title('Figure 1. Comparison of LLM Accuracy with Student Performance', fontsize=14)
plt.xlabel('Large Language Model', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(0, 100) # 0% から 100% の範囲に設定
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# 図をファイルに保存
figure1_path = os.path.join(save_dir, 'Figure_1.png')
plt.savefig(figure1_path, bbox_inches='tight', dpi=300)
plt.close()

print(f"Figure 1 saved to {figure1_path}")

/tmp/ipykernel_6875/1465428696.py:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Model', y='Accuracy (%)', data=llm_accuracy_df, palette='viridis')


Figure 1 saved to ./save_figures/Figure_1.png


### Example: 受験生平均とLLMの比較（挿入例）

以下は受験生平均や標準偏差をプロットした見本画像です。自分で作成した Figure 1 は `./save_figures/Figure_1.png` として保存してください。

![精度分布グラフ](https://github.com/Nakaura-T/DS_Seminar3_Public/blob/main/notebooks/images/model_performance_distribution.png?raw=1)

## 5) 各分野のLLM正解率をプロットしてレーダーチャートをつくる

目標: 各 `Subspeciality` ごとにモデルの正解率を集計し、モデルごとにレーダーチャートで比較します。

次の手順を参考に進めてください。
- `by_field = df.groupby('Subspeciality')[models].mean()` で分野別正答率を求めます。
- レーダーチャートでは各分野を角度にマッピングし、最後の点を先頭に追加してループを閉じます。
- matplotlib の `projection='polar'` を使って複数モデルを重ねて描画します。
- 軸のレンジを 0-100 (%) に揃えると見やすくなります。
- 作成した Figure 2 は `./save_figures/Figure_2.png` として保存します。

In [22]:
# TODO: 分野別 accuracy のレーダーチャート（Figure 2）を作成してください
# ヒント: 角度配列を作成し、各モデルの分野別 accuracy を極座標で描画する
# ヒント: 凡例・目盛りを整えて保存（保存先: ./save_figures/Figure_2.png）


In [23]:
import numpy as np
import matplotlib.pyplot as plt
import os

# 'save_figures' ディレクトリが存在しない場合は作成
save_dir = './save_figures'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# モデルのリストを取得
models = subspecialty_accuracy.columns

# 分野のリストを取得
categories = list(subspecialty_accuracy.index)

# レーダーチャート用の角度を計算
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1] # 閉じた図形にするために最初の点を最後に追加

# プロットの作成
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# 各モデルのデータをプロット
for model in models:
    values = subspecialty_accuracy.loc[:, model].tolist()
    values += values[:1] # 閉じた図形にするために最初の点を最後に追加
    ax.plot(angles, values, linewidth=1, linestyle='solid', label=model)
    ax.fill(angles, values, alpha=0.25) # 塗りつぶし

# 目盛りの設定
ax.set_theta_offset(np.pi / 2) # 開始角度を上にする
ax.set_theta_direction(-1) # 時計回りにする

# カテゴリラベルの設定
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)

# Y軸の目盛りを0-100%に設定
ax.set_ylim(0, 100)
y_ticks = np.arange(0, 101, 20)
ax.set_yticks(y_ticks)
ax.set_yticklabels([f'{int(y)}%' for y in y_ticks])
ax.set_rlabel_position(0) # ラベル位置の調整

# タイトルと凡例
plt.title('Figure 2. Subspecialty-specific Accuracy of LLMs', size=14, color='black', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# 図をファイルに保存
figure2_path = os.path.join(save_dir, 'Figure_2.png')
plt.savefig(figure2_path, bbox_inches='tight', dpi=300)
plt.close(fig)

print(f"Figure 2 saved to {figure2_path}")

print("\nSubspecialty Accuracy Data:")
display(subspecialty_accuracy.round(1))

Figure 2 saved to ./save_figures/Figure_2.png

Subspecialty Accuracy Data:


,GPT-4.1,o3,Claude 3.7 Sonnet,Claude 3.7 Sonnet-thinking,Gemini 2.5 Pro Preview,Gemini 2.5 Flash Preview-thinking
Subspeciality,,,,,,
Abdominal & Pelvic,51.9,77.8,51.9,51.9,74.1,63.0
MSK & Breast,63.6,45.5,63.6,72.7,63.6,63.6
Neuroradiology & Head & Neck,68.8,68.8,50.0,43.8,56.2,43.8
Nuclear Medicine,60.0,70.0,50.0,55.0,75.0,60.0
Other,66.7,83.3,83.3,100.0,100.0,100.0
Thoracic & Cardiac,75.0,80.0,75.0,65.0,85.0,90.0


### Example: レーダーチャートの挿入例

以下は分野ごとのモデル正解率を比較したレーダーチャートの見本画像です。自分で作成した Figure 2 は `./save_figures/Figure_2.png` として保存してください。

![レーダーチャート](https://github.com/Nakaura-T/DS_Seminar3_Public/blob/main/notebooks/images/radar_chart.png?raw=1)

## 6) Markdown で表・図へリンクする

この実習では、作成した表・図を `./save_figures` に保存し、論文ドラフトの Markdown からリンクします。Markdown では、画像を表示したい場所に次の形式で書きます。

```markdown
![表示名](画像ファイルへの相対パス)
```

このノートブックと同じ場所に `save_figures` フォルダがある場合、以下のように書きます。ファイル名には空白を使わず、`_` を使うと Pandoc で DOCX に変換するときも安全です。

```markdown
![Table 1](./save_figures/Table_1.png)
![Table 2](./save_figures/Table_2.png)
![Figure 1](./save_figures/Figure_1.png)
![Figure 2](./save_figures/Figure_2.png)
```

論文ドラフトでは、画像リンクの直前または直後に Table Legend / Figure Legend を書きます。例:

```markdown
Table 1. Distribution of radiology board examination questions by subspeciality.

![Table 1](./save_figures/Table_1.png)

Figure 1. Comparison of LLM accuracy with examinee performance.

![Figure 1](./save_figures/Figure_1.png)
```

注意: `./images` は見本画像を見るためのフォルダです。自分で作成した提出用の表・図は `./save_figures` からリンクしてください。

### Pandoc で DOCX に変換する

この実習では PDF ではなく DOCX に出力します。Pandoc で DOCX に変換する場合、LaTeX は不要です。Markdown ファイルと `save_figures` フォルダの位置関係が正しければ、画像は Word ファイル内に埋め込まれます。

```bash
pandoc paper.md -o paper.docx
```

Markdown ファイルが `notebooks` フォルダ内にあり、`notebooks/save_figures` を参照する場合は、次のような構成にしてください。

```text
notebooks/
├── paper.md
├── save_figures/
│   ├── Table_1.png
│   ├── Table_2.png
│   ├── Figure_1.png
│   └── Figure_2.png
```

本文中では、図表番号は手動で `Table 1`, `Table 2`, `Figure 1`, `Figure 2` と書いて参照します。

```markdown
The distribution of questions is shown in Table 1.

Table 1. Distribution of radiology board examination questions by subspeciality.

![Table 1](./save_figures/Table_1.png)
```

### 図表の自動参照について

`pandoc-crossref` を使うと `@fig:...` や `@tbl:...` のような自動参照も可能ですが、授業では環境差を避けるため使いません。まずは本文中に `Table 1` / `Figure 1` と手動で書く方法に統一します。

### DOCX での改ページ

DOCX では、改ページは変換後に Word で調整するのが簡単です。Markdown 内で改ページを指定したい場合は、Pandoc の DOCX 出力では OpenXML を使う方法があります。

```markdown
# Results
（本文）

~~~{=openxml}
<w:p><w:r><w:br w:type="page"/></w:r></w:p>
~~~

Table 1. Distribution of radiology board examination questions by subspeciality.

![Table 1](./save_figures/Table_1.png)
```

ただし、最初は改ページを入れずに DOCX を作成し、Word で最終調整する方が安全です。


## 7) Table Legend を書く

### Table Legend とは

Table Legend（表の説明文、caption）は、読者が本文を読まなくても「この表が何を示しているか」を理解できるようにする短い説明です。表そのものに書ききれない補足情報を整理して書きます。

Table Legend に含める内容:
- 表番号と短いタイトル: `Table 1. Distribution of questions by subspeciality.` のように書きます。
- 対象と単位: 何を数えた表か、`n`、`%`、accuracy などの単位を明記します。
- 略語の説明: LLM、SD など、表だけを見た読者に必要な略語を定義します。
- 計算方法の補足: accuracy を「正解数 / 問題数」として計算した、などを簡潔に書きます。

注意点:
- Legend は結果の解釈を長く述べる場所ではありません。
- 「どのモデルが最も優れていた」などの主張は Results 本文に書きます。
- 表中のすべての略語は、本文で説明済みでも legend 内で再度説明するのが基本です。


### Exercise 7-1: Table 1 の Legend を書く

Table 1 は、各 `Subspeciality` に含まれる問題数を示す表です。次の空欄を自分のデータに合わせて埋めてください。

```markdown
Table 1. Distribution of radiology board examination questions by subspeciality.

The table shows the number of questions included in each subspeciality category. Values are presented as counts and percentages of the total number of questions (n = ___).
```

自分で書く欄:

```markdown
Table 1. Distribution of radiology board examination questions by subspeciality.

The table shows the number of questions included in each subspeciality category. Values are presented as counts and percentages of the total number of questions (n = ___).
```


### Exercise 7-2: Table 2 の Legend を書く

Table 2 は、各 LLM の全体 accuracy と分野別 accuracy を示す表です。accuracy の定義、単位、略語を含めて書いてください。

```markdown
Table 2. Overall and subspeciality-specific accuracy of large language models.

Accuracy was calculated as the percentage of correctly answered questions. Rows indicate models, and columns indicate overall accuracy and accuracy for each subspeciality. LLM = large language model.
```

自分で書く欄:

```markdown
Table 2. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


## 8) Figure Legend を書く

### Figure Legend とは

Figure Legend（図の説明文）は、図を見た読者が「何を比較しているのか」「軸や色が何を意味するのか」「エラーバーや線が何を示すのか」を理解できるようにする説明です。

Figure Legend に含める内容:
- 図番号と短いタイトル: `Figure 1. Comparison of model accuracy with student performance.` のように書きます。
- 図の内容: 何を、どのグループで、どの指標により比較したかを書きます。
- 軸・色・線・エラーバーの意味: `The dashed line indicates...` のように説明します。
- 略語の説明: LLM、SD などを定義します。

注意点:
- 図から読み取れる主要な傾向を少し書いてもよいですが、詳細な解釈は Results 本文に書きます。
- 図の legend は、読者が本文を読まなくても図を理解できる程度に具体的にします。


### Exercise 8-1: Figure 1 の Legend を書く

Figure 1 は、受験生平均と LLM の正解率を比較した図です。受験生の平均正解率と SD の具体的な数値、破線、帯など、自分の図で使った要素を説明してください。

```markdown
Figure 1. Comparison of LLM accuracy with student performance.

Bars show the overall accuracy of each LLM. The dashed horizontal line indicates the mean score of examinees (__%), and the shaded area indicates one standard deviation (SD, __%). LLM = large language model; SD = standard deviation.
```

自分で書く欄:

```markdown
Figure 1. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


### Exercise 8-2: Figure 2 の Legend を書く

Figure 2 は、分野別 accuracy をレーダーチャートで示した図です。各軸、線、塗りつぶし、単位を説明してください。

```markdown
Figure 2. Subspeciality-specific accuracy of LLMs.

The radar chart shows the accuracy of each LLM across subspeciality categories. Each axis represents one subspeciality, and values are shown as percentages. LLM = large language model.
```

自分で書く欄:

```markdown
Figure 2. ________________________________________________.

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


## 9) Results を書く

### Results とは

Results は「データから何が得られたか」を客観的に書くセクションです。ここでは、方法の詳しい説明や、なぜその結果になったかという深い考察は書きません。それらは Materials and Methods や Discussion に分けます。

Results に書く内容:
- 解析に含まれたデータの要約: 最終的に解析に含まれた問題数、分野数、モデル数を短く確認します。詳しいデータの説明や選択基準は Materials and Methods に書きます。
- 表や図で示した主要な数値: 全体 accuracy、分野別 accuracy、受験生平均との比較など、読者に伝えるべき代表的な数値を書きます。
- データから直接言える傾向: どのモデルが高い/低い、どの分野で差が大きい、受験生平均と比べてどうだったかを書きます。
- 表・図への参照: 本文中で `Table 1`, `Table 2`, `Figure 1`, `Figure 2` を必ず引用します。

表や図を引用するときの考え方:
- `Table 1` は、解析に含まれた問題の分野別内訳を示すときに引用します。
- `Table 2` は、モデルごとの全体 accuracy や分野別 accuracy の数値を示すときに引用します。
- `Figure 1` は、LLM と受験生平均を視覚的に比較するときに引用します。
- `Figure 2` は、分野ごとの性能差やモデル間のパターンを説明するときに引用します。
- 本文では、表や図のすべての数値を繰り返すのではなく、最も重要な結果だけを選んで書きます。

避けること:
- 「なぜそうなったか」の推測を長く書く。
- 表や図の数値をすべて文章で繰り返す。
- Table や Figure を置くだけで、本文中で引用しない。
- 主観的な表現を使う。例: 「すごく良い」「かなり悪い」など。


### Exercise 9: Results のドラフトを書く

以下のヒントをもとに、自分が作成した表・図の数値に合わせて Results を英語で書いてください。ここでは英文の完成例は示しません。数値は必ず自分の解析結果から入力してください。

書くときのヒント:
- 1段落目: 最終的に解析に含まれた問題数、分野数、モデル数を短く書く。分野別の問題数は `Table 1` を引用する。
- 2段落目: 全体 accuracy の結果を書く。最も高かったモデル、次に高かったモデル、最も低かったモデルを選び、`Table 2` を引用する。
- 3段落目: 受験生平均との比較を書く。受験生平均より高い/低い/同程度のモデルを述べ、`Figure 1` を引用する。
- 4段落目: 分野別 accuracy の結果を書く。分野によって性能差があったか、どの分野で高い/低い傾向があったかを述べ、`Table 2` と `Figure 2` を引用する。
- 各段落では、表や図の全数値を繰り返さず、本文で強調すべき主要な数値だけを選ぶ。
- Results では、なぜその差が出たかという解釈は詳しく書かない。解釈は Discussion に回す。

自分で書く欄:

```markdown
# Results

__________________________________________________________
__________________________________________________________
__________________________________________________________

__________________________________________________________
__________________________________________________________
__________________________________________________________

__________________________________________________________
__________________________________________________________
__________________________________________________________
```


# Results

This analysis included 100 multiple-choice questions covering 6 distinct subspecialties, evaluated across 6 large language models (LLMs). The distribution of questions by subspecialty is detailed in Table 1, showing that 'Abdominal & Pelvic' had the highest number of questions (n=27, 27.0%), followed by 'Thoracic & Cardiac' and 'Nuclear Medicine' (both n=20, 20.0%).

The overall accuracy of the LLMs ranged from 59.0% to 74.0%. Gemini 2.5 Pro Preview achieved the highest overall accuracy at 74.0%, followed by o3 at 72.0%. The lowest overall accuracies were observed for Claude 3.7 Sonnet and Claude 3.7 Sonnet-thinking, both at 59.0% (Table 2).

When compared to the hypothetical student mean performance of 72.9% (SD = 9.3%), Gemini 2.5 Pro Preview (74.0%) showed comparable performance. Other models like GPT-4.1 (63.0%), Claude 3.7 Sonnet (59.0%), and Claude 3.7 Sonnet-thinking (59.0%) performed below the student mean (Figure 1).

Subspecialty-specific accuracy demonstrated variability across both models and subspecialties. For instance, 'Other' showed a perfect 100.0% accuracy for Gemini 2.5 Flash Preview-thinking, while 'Neuroradiology & Head & Neck' was challenging for most models, with GPT-4.1 achieving 43.8%. These detailed performances across different subspecialties are presented in Table 2 and visualized in Figure 2.

## 10) Materials and Methods を書く

### Materials and Methods とは

Materials and Methods は「誰かが同じ解析を再現できるように、何を使って、どのように解析したか」を書くセクションです。Results とは異なり、ここでは結果の数値や解釈ではなく、手順と条件を明確に書きます。

まず、自然科学論文全般で一般に必要とされる要素を確認します。

- `Study design / Ethical considerations`:
  研究デザイン（実験・観察・シミュレーション等）、対象、倫理審査・同意・公開データ利用の扱い。
- `Materials / Data sources`:
  試料・機器・データの出所、選定基準、サンプル数、除外基準。
- `Procedures / Protocol`:
  実験手順・測定条件・前処理・品質管理（QC）・再現に必要な設定。
- `Outcome definitions`:
  何を主要評価項目としたか、変数や指標の定義（単位・計算式を含む）。
- `Statistical analysis`:
  使用した統計手法、仮説検定、信頼区間、有意水準、ソフトウェアとバージョン。
- `Reproducibility information`:
  コード・データ・乱数シード・実行環境など再現性に必要な情報。

このような一般指針は実在します。代表例として、医学系は EQUATOR Network（CONSORT, STROBE, PRISMA など）、生命科学は ARRIVE などの報告ガイドラインがあります。

この演習では、`MATERIALS AND METHODS` を以下の5項目で書いてください。

- `Ethical Considerations`
- `Data Collection`
- `Model Selection`
- `MLLM Interrogation and Data Processing`
- `Statistical Analysis`

各項目を書くために必要な情報を先に整理しておきます。

- `Ethical Considerations`:
  公開されている試験問題を使用し、人・動物対象研究は実施していないため倫理審査は不要。データは匿名化され、関連するプライバシー規制・ガイドラインに従って取り扱った。

- `Data Collection`:
  データセットは 2024 年 JRS（日本医学放射線学会）2次試験の多肢選択問題 100 問。内訳は画像あり 96 問（96%）、テキストのみ 4 問（4%）。問題は日本語で、6分野（Abdominal/Pelvic, Thoracic/Cardiac, Nuclear Medicine, Neuroradiology/Head&Neck, MSK/Breast, Other）をカバー。問題PDFは JRS 会員向けサイトから取得し、Adobe Acrobat でテキストと画像を抽出。画像は主要画像を PNG 300 DPI で分離。人間受験者成績は JRS への照会で取得し、平均 72.9%、SD 9.26%。

- `Model Selection`:
  2025年春時点の最新モデルとして 6 モデルを選定。選定原則は、(1) OpenAI/Anthropic/Google の主要フラッグシップを含めること、(2) アーキテクチャや派生（thinking variant）に多様性を持たせること。対象モデルは GPT-4.1, o3, Claude 3.7 Sonnet, Claude 3.7 Sonnet-thinking, Gemini 2.5 Pro Preview, Gemini 2.5 Flash Preview-thinking。

- `MLLM Interrogation and Data Processing`:
  OpenRouter API を Python 3.11.7 + openai ライブラリ 1.63.2 で自動実行。各設問は再試行なしで1回だけ問い合わせ、ステートレスに独立処理。画像問題（96問）は「画像あり」と「画像なし」の2条件で評価。画像は無加工で PNG を Base64 化し、API の image URL フィールドへ data URI 形式で送信。回答は説明文を含む場合があるため、手動確認して単一選択肢（a-e）に正規化し、正答判定に使用。回答拒否（画像が必要等）は不正解として扱った。

- `Statistical Analysis`:
  主要評価項目は accuracy（正答率）。採点は strict exact match（完全一致のみ正解、部分点なし）。各モデルについて全体 accuracy と分野別 accuracy を算出し、人間受験者平均と比較。画像あり/なしの比較は同一 96 問のペアデータに対して McNemar 検定を適用。有意水準は p < 0.05。統計解析は Python 3.11.7 と SciPy 1.13.1 で実施。

書き方のポイント:
- 過去形で書く（例: `Accuracy was calculated ...`）。
- 再現に必要な情報を優先する。
- 結果の良し悪しの解釈は Results に書き、Methods には書かない。


### Exercise 10: Materials and Methods のドラフトを書く

以下のヒントをもとに、自分の解析内容に合わせて Materials and Methods を英語で書いてください。ここでは英文の完成例は示しません。

書くときのヒント（推奨項目順）:
- `Ethical Considerations`: 公開データ利用であること、倫理審査要否、匿名化の扱いを書く。
- `Data Collection`: 使用した CSV、問題数、分野数、画像あり/なしの内訳、主要カラムを書く。
- `Model Selection`: 評価したモデル名、比較対象（student baseline）の扱い、選定理由を書く。
- `MLLM Interrogation and Data Processing`: 実行環境、前処理、プロンプト条件、回答整形ルール、保存先を簡潔に書く。
- `Statistical Analysis`: accuracy の定義、全体/分野別の計算、student baseline の平均・SD、必要なら検定法と有意水準を書く。
- Methods では結果の数値や「どのモデルが良かったか」という解釈は書かない。

自分で書く欄:

```markdown
# Materials and Methods

## Ethical Considerations
__________________________________________________________
__________________________________________________________
__________________________________________________________

## Data Collection
__________________________________________________________
__________________________________________________________
__________________________________________________________

## Model Selection
__________________________________________________________
__________________________________________________________
__________________________________________________________

## MLLM Interrogation and Data Processing
__________________________________________________________
__________________________________________________________
__________________________________________________________

## Statistical Analysis
__________________________________________________________
__________________________________________________________
__________________________________________________________
```


# Materials and Methods

## Ethical Considerations
This study utilized publicly available examination questions and did not involve human or animal subjects, thus ethical review was not required. All data were anonymized and handled in accordance with relevant privacy regulations and guidelines.

## Data Collection
The dataset comprised 100 multiple-choice questions from the 2024 JRS (Japan Radiological Society) Second Examination. The questions were in Japanese and covered six subspecialties: Abdominal & Pelvic, Thoracic & Cardiac, Nuclear Medicine, Neuroradiology & Head & Neck, MSK & Breast, and Other. Out of 100 questions, 96 (96%) included images, while 4 (4%) were text-only. Questions were extracted from PDF files obtained from the JRS member site using Adobe Acrobat, and primary images were separated as PNG files at 300 DPI. Student baseline performance data, with a mean of 72.9% and a standard deviation (SD) of 9.26%, was acquired through inquiry with the JRS.

## Model Selection
Six large language models (LLMs) were selected as of spring 2025. The selection criteria included: (1) incorporating flagship models from major providers (OpenAI, Anthropic, Google), and (2) ensuring diversity in architecture and derivative versions (e.g., 'thinking' variants). The evaluated models were GPT-4.1, o3, Claude 3.7 Sonnet, Claude 3.7 Sonnet-thinking, Gemini 2.5 Pro Preview, and Gemini 2.5 Flash Preview-thinking.

## MLLM Interrogation and Data Processing
Model interrogation was automated using the OpenRouter API via Python 3.11.7 and the `openai` library 1.63.2. Each question was queried once without retries, and processing was stateless. Image-based questions (96 questions) were evaluated under two conditions: with and without images. Images were converted to Base64 without modification and sent as data URI format in the API's image URL field. Model responses, which sometimes included explanatory text, were manually reviewed and normalized to a single choice (a-e) for correctness assessment. Responses that indicated an inability to answer (e.g., due to missing images) were treated as incorrect.

## Statistical Analysis
The primary outcome measure was accuracy, defined as the proportion of correctly answered questions. Scoring was based on a strict exact match criterion, with no partial credit. Overall accuracy and subspecialty-specific accuracy were calculated for each model and compared against the human examinee mean. For comparisons between image-present and image-absent conditions for the same 96 questions, McNemar's test was applied. A significance level of p < 0.05 was used. Statistical analyses were performed using Python 3.11.7 and SciPy 1.13.1.

## 11) チェックリスト

以下を確認してください。

- Table 1 と Table 2 に、それぞれ Table Legend がある。
- Table 1、Table 2、Figure 1、Figure 2 が `./save_figures` に保存されている。
- 論文ドラフトの Markdown で、`./save_figures/Table_1.png` などの相対パスを使って自分の表・図にリンクしている。
- Figure 1 と Figure 2 に、それぞれ Figure Legend がある。
- すべての legend で略語を説明している。
- Results では、Table 1、Table 2、Figure 1、Figure 2 を本文中で参照している。
- Results には主要な数値が入っている。
- Materials and Methods には、データ、前処理、accuracy の定義、分野別解析、可視化方法が書かれている。
- Results に方法の詳細を書きすぎていない。
- Materials and Methods に結果の解釈を書いていない。
- Materials and Methods と Results を `paper.md` にまとめて保存している。
- `pandoc` で `paper.docx` を作成し、Word で内容と画像の表示を確認している。


## 12) `paper.md` を保存して DOCX を作成する

最後に、今回作成した `Materials and Methods` と `Results` を 1 つの Markdown ファイルにまとめ、DOCX を作成します。

手順:
1. ノートブック内で書いた `Materials and Methods` と `Results` を `paper.md` として保存する（章の順番は `Materials and Methods` → `Results`）。
2. 表・図の相対パス（例: `./save_figures/Table_1.png`）が正しいことを確認する。
3. ターミナルで次を実行して DOCX を作成する。
4. 作成した `paper.docx` を Word で開き、表・図が表示されていることを確認する。

```bash
pandoc paper.md -o paper.docx
```

`paper.md` と `save_figures` は、次のような位置関係にしてください。

```text
notebooks/
├── paper.md
├── paper.docx
└── save_figures/
    ├── Table_1.png
    ├── Table_2.png
    ├── Figure_1.png
    └── Figure_2.png
```

`paper.md` の例:

```markdown
# Materials and Methods

## Ethical Considerations
（今回作成した文章）

## Data Collection
（今回作成した文章）

## Model Selection
（今回作成した文章）

## MLLM Interrogation and Data Processing
（今回作成した文章）

## Statistical Analysis
（今回作成した文章）

# Results

（今回作成した文章）

**Table 1. Distribution of questions by subspecialty**

![Table 1](./save_figures/Table_1.png)

**Table 2. Model accuracy (overall and by subspecialty)**

![Table 2](./save_figures/Table_2.png)

**Figure 1. Comparison between examinee baseline and model accuracy**

![Figure 1](./save_figures/Figure_1.png)

**Figure 2. Subspecialty-wise model performance (radar chart)**

![Figure 2](./save_figures/Figure_2.png)
```

DOCX では、改ページや細かいレイアウトは変換後に Word で調整するのが簡単です。Markdown 側ではまず、見出し、本文、Table/Figure legend、画像リンクが正しく並んでいることを優先してください。
